# SkyLens India — Exploratory Data Analysis

Analysis of `flight_date.xlsx` (10,683 rows) to understand the real dataset before building the pipeline.

In [ ]:
import pandas as pd
import re

df = pd.read_excel('flight_date.xlsx')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Data quality checks
print('Null counts:')
print(df.isnull().sum())
print()
print('Dtypes:')
print(df.dtypes)

In [ ]:
# Airline distribution — 12 carriers, Jet Airways dominates
print('Flights per airline:')
print(df['Airline'].value_counts())

In [ ]:
# Route analysis
print('Source cities:')  
print(df['Source'].value_counts())
print()
print('Destination cities:')
print(df['Destination'].value_counts())

In [ ]:
# Stop distribution
print('Stop counts (Total_Stops):')
print(df['Total_Stops'].value_counts())
# Key finding: 1-stop is majority (52.7%), non-stop is 32.7%

In [ ]:
# Price analysis
print('Price distribution (INR):')
print(df['Price'].describe())
# P25=5277, P50=8372, P75=12373, Max=79512 (outlier: Jet Airways Business multi-stop)

In [ ]:
# Duration parsing — real formats: '2h 50m', '19h', '21h 5m'
def parse_duration(s):
    h = re.search(r'(\d+)h', str(s))
    m = re.search(r'(\d+)m', str(s))
    return (int(h.group(1)) if h else 0)*60 + (int(m.group(1)) if m else 0)

df['duration_min'] = df['Duration'].apply(parse_duration)
print('Duration in minutes:')
print(df['duration_min'].describe())
# Min=75 (1h15m), Max=1770 (29h30m) — multi-stop with long layover

In [ ]:
# Data quality issue: next-day arrivals encoded in Arrival_Time
next_day = df[df['Arrival_Time'].str.contains('Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec', na=False)]
print(f'Rows with next-day arrival date in Arrival_Time: {len(next_day)} ({100*len(next_day)/len(df):.1f}%)')
print(next_day[['Airline','Source','Destination','Dep_Time','Arrival_Time','Duration']].head(8))

In [ ]:
# Average price by airline — key insight for dashboard
print('Average price by airline (INR):')
print(df.groupby('Airline')['Price'].mean().sort_values(ascending=False).round(0))

In [ ]:
# Price vs stops — do more stops = cheaper?
print('Average price by stop count:')
print(df.groupby('Total_Stops')['Price'].mean().sort_values().round(0))
# Insight: non-stop is often MORE expensive (convenience premium)

In [ ]:
# Date parsing — inconsistent zero-padding: '1/05/2019' vs '24/03/2019'
df['journey_date_parsed'] = pd.to_datetime(df['Date_of_Journey'], dayfirst=True)
print('Date range:')
print(f"Min: {df['journey_date_parsed'].min()}")
print(f"Max: {df['journey_date_parsed'].max()}")
print()
print('Flights per month:')
print(df['journey_date_parsed'].dt.month.value_counts().sort_index())

In [ ]:
# City name issue: 'Banglore' is a misspelling of 'Bengaluru'
print('Source city values:', df['Source'].unique())
print('Destination values:', df['Destination'].unique())
# 'Banglore' appears — standardised to 'Bengaluru' in stg_flights.sql